# YOLO11n Indoor Navigation Training — Version 2

This notebook creates a clean and reproducible comparison model for the FYP indoor distance-to-haptic system. It keeps the same 27-class scope as the first experiment, but fixes mixed polygon/box annotations, preserves official COCO class names, balances COCO against the custom data, and evaluates a custom-only test split.

**Pipeline:** Roboflow v2 → class/split audit → clean boxes → balanced indoor COCO subset → YOLO11n transfer learning → merged validation + custom-only test → download `best.pt`.

> Select **Runtime → Change runtime type → T4 GPU** before starting. Run cells from top to bottom. Do not export TensorRT on Colab; export `best.pt` on the target Orin Nano.

In [ ]:
%pip install -q "ultralytics==8.4.138" "roboflow>=1.1,<2" pyyaml requests

import ultralytics, torch
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select a T4 GPU runtime and reconnect.')
print('GPU:', torch.cuda.get_device_name(0))

## Configuration

The COCO quota is deliberately smaller than version 1 so that generic COCO images do not dominate the custom indoor camera domain. Shared COCO labels retain their official spelling so Ultralytics can remap the pretrained class-head rows by name.

In [ ]:
from pathlib import Path

WORK_ROOT = Path('/content/fyp2_yolo11n_v2')
CUSTOM_DIR = WORK_ROOT / 'roboflow_v2'
COCO_CACHE = WORK_ROOT / 'coco_cache'
MERGED_DIR = WORK_ROOT / 'indoor_coco_merged_v2'
RUNS_DIR = WORK_ROOT / 'runs'
RUN_NAME = 'yolo11n_indoor_v2'

ROBOFLOW_WORKSPACE = 'test-workspace-dwuie'
ROBOFLOW_PROJECT = 'my-second-project-person-chair-add-chair'
ROBOFLOW_VERSION = 2

EXPECTED_CUSTOM_CLASSES = [
    'chair', 'cleaning_cart', 'handrail', 'person', 'pillars',
    'ramp', 'squat-toilet', 'stairs', 'urinals',
    'vending_machine', 'wet_floor_sign',
]
COCO_CLASSES = [
    'person', 'bicycle', 'chair', 'bench', 'cat', 'dog', 'backpack',
    'umbrella', 'handbag', 'suitcase', 'bottle', 'bed',
    'dining table', 'couch', 'toilet', 'potted plant', 'sink',
    'refrigerator',
]
INDOOR_ANCHORS = {
    'chair', 'bed', 'dining table', 'couch', 'toilet',
    'potted plant', 'sink', 'refrigerator',
}
FINAL_CLASSES = EXPECTED_CUSTOM_CLASSES + [
    name for name in COCO_CLASSES if name not in EXPECTED_CUSTOM_CLASSES
]
assert len(FINAL_CLASSES) == 27, FINAL_CLASSES

TRAIN_PER_COCO_CLASS = 250
VAL_PER_COCO_CLASS = 50
MAX_COCO_TRAIN_IMAGES = 2500
MAX_COCO_VAL_IMAGES = 500
SEED = 42

EPOCHS = 120
PATIENCE = 25
IMGSZ = 640
BATCH = 16
WORKERS = 2

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('Final classes:', len(FINAL_CLASSES))
for index, name in enumerate(FINAL_CLASSES):
    print(f'{index:2d}: {name}')

## Download Roboflow version 2

The API key is entered through a hidden prompt and is not stored in this notebook. YOLOv8 and YOLO11 use the same detection text-label layout, so the established Roboflow `yolov8` exporter is used.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

if not (CUSTOM_DIR / 'data.yaml').is_file():
    api_key = getpass('Roboflow private API key: ')
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    downloaded = version.download(
        'yolov8', location=str(CUSTOM_DIR), overwrite=True
    )
    print('Downloaded:', downloaded.location)
else:
    print('Using existing download:', CUSTOM_DIR)

## Validate class mapping and data splits

This cell reads the real Roboflow class order instead of assuming it. Minor space/hyphen/underscore spelling differences are normalized, while missing or unexpected classes stop the build. If a source-image name appears in multiple splits, the notebook preserves test first and validation second, then removes matches from lower-priority splits to prevent optimistic validation.

In [ ]:
import re
import yaml

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def normalized_name(name):
    return re.sub(r'[^a-z0-9]+', '_', str(name).lower()).strip('_')

raw_yaml = yaml.safe_load((CUSTOM_DIR / 'data.yaml').read_text())
raw_names = raw_yaml.get('names')
if isinstance(raw_names, dict):
    source_names = [raw_names[key] for key in sorted(raw_names, key=int)]
elif isinstance(raw_names, list):
    source_names = raw_names
else:
    raise RuntimeError('Roboflow data.yaml has no valid names list')

expected_by_normalized = {
    normalized_name(name): name for name in EXPECTED_CUSTOM_CLASSES
}
canonical_source_names = []
for source_name in source_names:
    canonical = expected_by_normalized.get(normalized_name(source_name))
    if canonical is None:
        raise RuntimeError(f'Unexpected Roboflow class: {source_name!r}')
    canonical_source_names.append(canonical)
if set(canonical_source_names) != set(EXPECTED_CUSTOM_CLASSES):
    missing = sorted(set(EXPECTED_CUSTOM_CLASSES) - set(canonical_source_names))
    raise RuntimeError(f'Roboflow classes are incomplete; missing={missing}')
if len(set(canonical_source_names)) != len(canonical_source_names):
    raise RuntimeError('Multiple Roboflow classes normalize to the same class')

source_id_to_final = {
    source_id: FINAL_CLASSES.index(name)
    for source_id, name in enumerate(canonical_source_names)
}
print('Roboflow source order:', canonical_source_names)
print('Source-to-final IDs:', source_id_to_final)

def source_key(path):
    return path.stem.split('.rf.')[0].lower()

split_folders = {'train': 'train', 'valid': 'val', 'test': 'test'}
keys_by_split = {}
for source_split in split_folders:
    image_dir = CUSTOM_DIR / source_split / 'images'
    if not image_dir.is_dir():
        raise FileNotFoundError(f'Missing Roboflow folder: {image_dir}')
    keys_by_split[source_split] = {
        source_key(path) for path in image_dir.iterdir()
        if path.suffix.lower() in IMAGE_SUFFIXES
    }
overlap_pairs = {}
for left, right in [('train', 'valid'), ('train', 'test'), ('valid', 'test')]:
    overlap = keys_by_split[left] & keys_by_split[right]
    overlap_pairs[f'{left}-{right}'] = overlap
    if overlap:
        print(
            f'WARNING: {len(overlap)} source keys overlap between {left} and {right}; '
            f'examples={sorted(overlap)[:10]}'
        )
excluded_source_keys = {
    'train': keys_by_split['valid'] | keys_by_split['test'],
    'valid': keys_by_split['test'],
    'test': set(),
}
print('Automatic split precedence: test > valid > train')
for split, excluded in excluded_source_keys.items():
    matching = keys_by_split[split] & excluded
    percentage = 100.0 * len(matching) / max(1, len(keys_by_split[split]))
    print(f'  {split}: {len(matching)} source keys will be removed ({percentage:.1f}%)')
CUSTOM_COCO_FILE_STEMS = set()
for keys in keys_by_split.values():
    for key in keys:
        candidate = key[:-4] if key.endswith('_jpg') else key
        if len(candidate) == 12 and candidate.isdigit():
            CUSTOM_COCO_FILE_STEMS.add(candidate)
print(f'COCO-named custom sources excluded from added COCO: {len(CUSTOM_COCO_FILE_STEMS)}')

## Build the clean merged dataset

Detection rows are preserved. Polygon rows are converted to bounding boxes. Class IDs are explicitly remapped from the downloaded Roboflow order. COCO contributes only relevant objects from indoor-proxy images; the test split remains custom-only.

In [ ]:
import json
import random
import shutil
import zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

COCO_S3_BASE = 'https://s3.us-east-1.amazonaws.com/images.cocodataset.org/'
ANNOTATIONS_URL = COCO_S3_BASE + 'annotations/annotations_trainval2017.zip'

def download_file(url, destination):
    if destination.is_file() and destination.stat().st_size > 0:
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.part')
    last_error = None
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=60) as response:
                response.raise_for_status()
                with temporary.open('wb') as output:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            output.write(chunk)
            temporary.replace(destination)
            return
        except Exception as error:
            last_error = error
            temporary.unlink(missing_ok=True)
    raise RuntimeError(f'Failed to download {url}: {last_error}')

def ensure_coco_annotations():
    archive = COCO_CACHE / 'annotations_trainval2017.zip'
    download_file(ANNOTATIONS_URL, archive)
    outputs = {}
    with zipfile.ZipFile(archive) as bundle:
        for split in ('train2017', 'val2017'):
            destination = COCO_CACHE / 'annotations' / f'instances_{split}.json'
            outputs[split] = destination
            if not destination.is_file():
                destination.parent.mkdir(parents=True, exist_ok=True)
                member = f'annotations/instances_{split}.json'
                with bundle.open(member) as source, destination.open('wb') as target:
                    shutil.copyfileobj(source, target)
    return outputs

def select_coco_images(coco, per_class_quota, max_images, seed):
    category_name = {item['id']: item['name'] for item in coco['categories']}
    target_ids = {cid for cid, name in category_name.items() if name in COCO_CLASSES}
    anchor_ids = {cid for cid, name in category_name.items() if name in INDOOR_ANCHORS}
    annotations_by_image = defaultdict(list)
    categories_by_image = defaultdict(set)
    for annotation in coco['annotations']:
        if annotation.get('iscrowd', 0):
            continue
        annotations_by_image[annotation['image_id']].append(annotation)
        categories_by_image[annotation['image_id']].add(annotation['category_id'])
    images = {item['id']: item for item in coco['images']}
    eligible = sorted(
        image_id for image_id, categories in categories_by_image.items()
        if categories & target_ids
        and categories & anchor_ids
        and Path(images[image_id]['file_name']).stem not in CUSTOM_COCO_FILE_STEMS
    )
    candidates = {
        category_id: [
            image_id for image_id in eligible
            if category_id in categories_by_image[image_id]
        ]
        for category_id in target_ids
    }
    rng = random.Random(seed)
    for values in candidates.values():
        rng.shuffle(values)
    selected, counts = set(), Counter()
    ordering = sorted(target_ids, key=lambda cid: (len(candidates[cid]), cid))
    for category_id in ordering:
        for image_id in candidates[category_id]:
            if counts[category_id] >= per_class_quota:
                break
            if image_id in selected:
                continue
            if len(selected) >= max_images:
                break
            selected.add(image_id)
            for present_id in categories_by_image[image_id] & target_ids:
                counts[present_id] += 1
    return sorted(selected), annotations_by_image, images, category_name, counts

def clean_custom_row(line, label_path):
    parts = line.split()
    if not parts:
        return None, False
    try:
        source_id = int(parts[0])
        coordinates = [float(value) for value in parts[1:]]
    except ValueError as error:
        raise ValueError(f'Invalid label row in {label_path}: {line}') from error
    if source_id not in source_id_to_final:
        raise ValueError(f'Invalid class {source_id} in {label_path}')
    converted_polygon = False
    if len(coordinates) == 4:
        center_x, center_y, width, height = coordinates
    elif len(coordinates) >= 6 and len(coordinates) % 2 == 0:
        xs, ys = coordinates[0::2], coordinates[1::2]
        left, right = min(xs), max(xs)
        top, bottom = min(ys), max(ys)
        center_x, center_y = (left + right) / 2, (top + bottom) / 2
        width, height = right - left, bottom - top
        converted_polygon = True
    else:
        raise ValueError(f'Unsupported label row in {label_path}: {line}')
    center_x = min(1.0, max(0.0, center_x))
    center_y = min(1.0, max(0.0, center_y))
    width = min(1.0, max(0.0, width))
    height = min(1.0, max(0.0, height))
    if width <= 0 or height <= 0:
        raise ValueError(f'Zero-area label in {label_path}: {line}')
    final_id = source_id_to_final[source_id]
    cleaned = f'{final_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}'
    return cleaned, converted_polygon

def add_custom_data():
    statistics = Counter()
    for source_split, output_split in split_folders.items():
        image_dir = CUSTOM_DIR / source_split / 'images'
        label_dir = CUSTOM_DIR / source_split / 'labels'
        for image_path in sorted(image_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            if source_key(image_path) in excluded_source_keys[source_split]:
                statistics[f'deduplicated_{source_split}_images'] += 1
                continue
            output_stem = f'custom__{image_path.stem}'
            output_image = MERGED_DIR / 'images' / output_split / (output_stem + image_path.suffix.lower())
            output_label = MERGED_DIR / 'labels' / output_split / (output_stem + '.txt')
            shutil.copy2(image_path, output_image)
            source_label = label_dir / f'{image_path.stem}.txt'
            cleaned_rows, seen = [], set()
            if source_label.is_file():
                for line in source_label.read_text().splitlines():
                    if not line.strip():
                        continue
                    cleaned, converted = clean_custom_row(line, source_label)
                    if cleaned in seen:
                        statistics['duplicate_rows_removed'] += 1
                        continue
                    seen.add(cleaned)
                    cleaned_rows.append(cleaned)
                    statistics['polygons_converted'] += int(converted)
                    statistics['custom_boxes'] += 1
            else:
                statistics['missing_labels_treated_as_background'] += 1
            output_label.write_text(('\n'.join(cleaned_rows) + '\n') if cleaned_rows else '')
            statistics[f'custom_{output_split}_images'] += 1
    return statistics

def coco_box_to_yolo(annotation, width, height):
    x, y, box_width, box_height = annotation['bbox']
    left = max(0.0, min(float(width), float(x)))
    top = max(0.0, min(float(height), float(y)))
    right = max(0.0, min(float(width), float(x + box_width)))
    bottom = max(0.0, min(float(height), float(y + box_height)))
    if right <= left or bottom <= top:
        return None
    return (
        ((left + right) / 2) / width, ((top + bottom) / 2) / height,
        (right - left) / width, (bottom - top) / height,
    )

def add_coco_data(split, output_split, selection):
    ids, annotations_by_image, images, category_name, _ = selection
    final_id = {name: index for index, name in enumerate(FINAL_CLASSES)}
    image_cache = COCO_CACHE / 'images' / split
    jobs = []
    for image_id in ids:
        image = images[image_id]
        relative_url = image['coco_url'].split('images.cocodataset.org/', 1)[-1]
        url = COCO_S3_BASE + relative_url
        jobs.append((url, image_cache / image['file_name']))
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(download_file, url, path): url for url, path in jobs}
        for number, future in enumerate(as_completed(futures), 1):
            future.result()
            if number % 250 == 0 or number == len(futures):
                print(f'{split}: downloaded/verified {number}/{len(futures)}')
    count = 0
    for image_id in ids:
        image = images[image_id]
        source_image = image_cache / image['file_name']
        output_stem = f'coco2017__{Path(image["file_name"]).stem}'
        output_image = MERGED_DIR / 'images' / output_split / (output_stem + source_image.suffix.lower())
        output_label = MERGED_DIR / 'labels' / output_split / (output_stem + '.txt')
        rows = []
        for annotation in annotations_by_image[image_id]:
            name = category_name[annotation['category_id']]
            if name not in COCO_CLASSES:
                continue
            box = coco_box_to_yolo(annotation, image['width'], image['height'])
            if box is None:
                continue
            rows.append(f'{final_id[name]} ' + ' '.join(f'{value:.6f}' for value in box))
            count += 1
        if not rows:
            raise RuntimeError(f'COCO image {image_id} produced no selected labels')
        shutil.copy2(source_image, output_image)
        output_label.write_text('\n'.join(rows) + '\n')
    return count

if MERGED_DIR.exists():
    if MERGED_DIR.parent != WORK_ROOT:
        raise RuntimeError('Refusing to replace a merged directory outside WORK_ROOT')
    shutil.rmtree(MERGED_DIR)
for folder in ('images', 'labels'):
    for split in ('train', 'val', 'test'):
        (MERGED_DIR / folder / split).mkdir(parents=True, exist_ok=True)

custom_statistics = add_custom_data()
print('Custom cleaning:', dict(custom_statistics))
annotation_paths = ensure_coco_annotations()
selections = {}
for offset, split in enumerate(('train2017', 'val2017')):
    coco = json.loads(annotation_paths[split].read_text())
    quota = TRAIN_PER_COCO_CLASS if split == 'train2017' else VAL_PER_COCO_CLASS
    maximum = MAX_COCO_TRAIN_IMAGES if split == 'train2017' else MAX_COCO_VAL_IMAGES
    selections[split] = select_coco_images(coco, quota, maximum, SEED + offset)
    print(split, 'selected images:', len(selections[split][0]))
coco_train_boxes = add_coco_data('train2017', 'train', selections['train2017'])
coco_val_boxes = add_coco_data('val2017', 'val', selections['val2017'])

data_yaml = {
    'path': str(MERGED_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(FINAL_CLASSES),
    'names': {index: name for index, name in enumerate(FINAL_CLASSES)},
}
(MERGED_DIR / 'data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print('COCO boxes:', {'train': coco_train_boxes, 'val': coco_val_boxes})
print('Dataset written to:', MERGED_DIR)

## Final dataset audit

Every output label must now have exactly five values: class ID plus one bounding box. The printed per-class counts make severe class imbalance visible before spending GPU time.

In [ ]:
dataset_report = {}
for split in ('train', 'val', 'test'):
    image_dir = MERGED_DIR / 'images' / split
    label_dir = MERGED_DIR / 'labels' / split
    images = [path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES]
    counts = Counter()
    backgrounds = 0
    for label_path in label_dir.glob('*.txt'):
        rows = [line for line in label_path.read_text().splitlines() if line.strip()]
        backgrounds += int(not rows)
        for line in rows:
            parts = line.split()
            if len(parts) != 5:
                raise RuntimeError(f'Non-detection row remains in {label_path}: {line}')
            class_id = int(parts[0])
            if not 0 <= class_id < len(FINAL_CLASSES):
                raise RuntimeError(f'Invalid class ID in {label_path}: {class_id}')
            counts[FINAL_CLASSES[class_id]] += 1
    if len(images) != len(list(label_dir.glob('*.txt'))):
        raise RuntimeError(f'Image/label count mismatch in {split}')
    dataset_report[split] = {
        'images': len(images), 'instances': sum(counts.values()),
        'backgrounds': backgrounds, 'class_instances': dict(counts),
    }
    print(f'\n{split.upper()}: {len(images)} images, {sum(counts.values())} boxes, {backgrounds} backgrounds')
    for name in FINAL_CLASSES:
        print(f'  {name:20s} {counts[name]:6d}')
if dataset_report['test']['images'] == 0:
    raise RuntimeError('A non-empty custom-only test split is required')
manifest = {
    'seed': SEED, 'classes': FINAL_CLASSES, 'report': dataset_report,
    'coco_limits': {
        'train_per_class': TRAIN_PER_COCO_CLASS,
        'val_per_class': VAL_PER_COCO_CLASS,
        'max_train_images': MAX_COCO_TRAIN_IMAGES,
        'max_val_images': MAX_COCO_VAL_IMAGES,
    },
}
(MERGED_DIR / 'build_manifest.json').write_text(json.dumps(manifest, indent=2))
print('\nDataset audit passed.')

## Train from COCO-pretrained YOLO11n

This remains transfer learning. Mild geometric and colour augmentation is used for indoor camera variation, while mosaic is reduced compared with the original run. If the run folder already exists, change `RUN_NAME` in the configuration cell to keep experiments separate.

In [ ]:
from ultralytics import YOLO

run_directory = RUNS_DIR / RUN_NAME
if run_directory.exists():
    raise FileExistsError(
        f'{run_directory} already exists. Change RUN_NAME before retraining.'
    )
model = YOLO('yolo11n.pt', task='detect')
model.train(
    data=str(MERGED_DIR / 'data.yaml'),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=WORKERS,
    pretrained=True,
    amp=True,
    optimizer='auto',
    cos_lr=True,
    seed=SEED,
    deterministic=True,
    degrees=5.0,
    translate=0.10,
    scale=0.35,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.30,
    mosaic=0.50,
    close_mosaic=10,
    mixup=0.0,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,
    plots=True,
    val=True,
    save=True,
)
BEST_PT = Path(model.trainer.best)
print('Best checkpoint:', BEST_PT)

## Evaluate merged validation and custom-only test

The custom-only test result is the more relevant number for your project. It still does not replace a separate field test recorded with the rectified IMX219 camera in previously unseen rooms.

In [ ]:
best_model = YOLO(str(BEST_PT), task='detect')

val_metrics = best_model.val(
    data=str(MERGED_DIR / 'data.yaml'), split='val', imgsz=IMGSZ,
    device=0, plots=True, project=str(RUNS_DIR), name=RUN_NAME + '_val',
)
test_metrics = best_model.val(
    data=str(MERGED_DIR / 'data.yaml'), split='test', imgsz=IMGSZ,
    device=0, plots=True, project=str(RUNS_DIR), name=RUN_NAME + '_custom_test',
)

def metric_summary(metrics):
    return {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
    }

evaluation = {
    'merged_validation': metric_summary(val_metrics),
    'custom_only_test': metric_summary(test_metrics),
}
print(json.dumps(evaluation, indent=2))
(run_directory / 'fyp2_evaluation_summary.json').write_text(
    json.dumps(evaluation, indent=2)
)

## Visual inspection of custom test predictions

Metrics alone do not show unstable boxes or obvious false detections. This cell saves predictions from up to 100 custom test images for manual inspection.

In [ ]:
test_images = sorted(
    path for path in (MERGED_DIR / 'images' / 'test').iterdir()
    if path.suffix.lower() in IMAGE_SUFFIXES
)[:100]
prediction_results = best_model.predict(
    source=[str(path) for path in test_images],
    imgsz=IMGSZ, conf=0.25, iou=0.50, device=0,
    save=True, project=str(RUNS_DIR), name=RUN_NAME + '_test_predictions',
    verbose=False,
)
print('Saved prediction images:', len(prediction_results))

## Optional fair comparison with the old `best.pt`

Set the flag to `True`, rerun the cell, and upload the old 6.0 `best.pt`. Both checkpoints will then be tested on exactly the same cleaned custom test set.

In [ ]:
RUN_OLD_MODEL_COMPARISON = False

if RUN_OLD_MODEL_COMPARISON:
    from google.colab import files
    uploaded = files.upload()
    old_path = Path('/content') / next(iter(uploaded))
    old_model = YOLO(str(old_path), task='detect')
    old_metrics = old_model.val(
        data=str(MERGED_DIR / 'data.yaml'), split='test', imgsz=IMGSZ,
        device=0, plots=True, project=str(RUNS_DIR),
        name='old_model_custom_test',
    )
    print('Old model:', json.dumps(metric_summary(old_metrics), indent=2))
    print('New model:', json.dumps(evaluation['custom_only_test'], indent=2))
else:
    print('Old-model comparison skipped.')

## Download artifacts

Download `best.pt` and the experiment archive. Keep the old checkpoint until the new model wins on the custom-only test and on real rectified IMX219 recordings.

In [ ]:
from google.colab import files

download_pt = WORK_ROOT / 'best_indoor_v2.pt'
shutil.copy2(BEST_PT, download_pt)
archive_base = WORK_ROOT / 'yolo11n_indoor_v2_results'
archive_path = Path(shutil.make_archive(
    str(archive_base), 'zip', root_dir=RUNS_DIR
))
print('Downloading:', download_pt, archive_path)
files.download(str(download_pt))
files.download(str(archive_path))

## Export on the Orin Nano—not Colab

Copy the downloaded checkpoint to the Jetson and build TensorRT on the exact deployment device:

```bash
cd /home/orin_nano/Desktop/FYP2/6.0_yolomodel
# Put best_indoor_v2.pt in this folder first.
/usr/bin/python3 export_best_pt_to_engine.py \
  --model best_indoor_v2.pt --imgsz 640 --batch 1
```

Do not replace the 7.0 engine until the new checkpoint has better custom-test results and stable detections on real IMX219 video.